# Detecció d'errors de lectura amb ASR

**Objectiu**: transcriure automàticament les gravacions de lectura amb el model **Whisper** i alinear cada paraula reconeguda amb el text de referència ("Los okapis") per identificar els errors de lectura de cada alumne.

L'alineació es fa amb **programació dinàmica** (distància d'edició): per a cada paraula del text de referència es determina l'operació corresponent (`correct`, `substitution`, `deletion` o `insertion`) i se'n recupera el temps de pronúncia a partir de les marques temporals de Whisper.

**Entrada**: àudios de lectura + text de referència (`text.txt`).  
**Sortida**: un fitxer CSV per alumne amb l'alineació paraula a paraula (operació d'edició + temps), reaprofitat per `04_errors.ipynb`.

> Aquest notebook s'executa a part de la resta del *pipeline*, ja que requereix paquets addicionals (Whisper, torch). Les transcripcions ja estan generades a `transcriptions/`.

## 1. Instal·lació de dependències

Instal·lem Whisper i les llibreries de suport (alineació temporal i distància de Levenshtein).

In [ ]:
%pip install torch openai-whisper==20230124 whisper-timestamped python-Levenshtein

## 2. Imports

Importem les llibreries per al maneig de fitxers, expressions regulars, distància d'edició i la transcripció amb marques temporals (`whisper_timestamped`).

In [ ]:
import os
import re
import json
import Levenshtein
import csv
import whisper_timestamped as whisper

## 3. Normalització del text

La funció `normalitza_text` passa el text a minúscules, elimina la puntuació i el divideix en una llista de paraules. S'aplica tant al text de referència com a la transcripció per fer-los comparables.

In [ ]:
def normalitza_text(text: str):
    text = text.lower()
    text = re.sub(r"[^a-zà-úçñ0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split(" ") if text else []

## 4. Alineació paraula a paraula (distància d'edició)

La funció `align_words_tokens` calcula l'alineació òptima entre les paraules de referència i les reconegudes mitjançant programació dinàmica. Retorna, per a cada posició, l'operació d'edició (`correct`, `substitution`, `deletion`, `insertion`).

In [ ]:
def align_words_tokens(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1): dp[i][m] = n - i + 1
    for j in range(1, m + 1): dp[n][j] = m - j + 1

    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            cost_sub = dp[i + 1][j + 1] if ref[i] == hyp[j] else dp[i + 1][j + 1] + 1
            dp[i][j] = min(cost_sub, dp[i + 1][j] + 1, dp[i][j + 1] + 1)

    i = j = 0
    ops = []
    while i < n or j < m:
        if i < n and j < m and ref[i] == hyp[j]:
            ops.append({"type": "correct", "ref": ref[i], "hyp": hyp[j]})
            i += 1; j += 1
        elif i < n and j < m and dp[i][j] == dp[i+1][j+1] + 1:
            ops.append({"type": "substitution", "ref": ref[i], "hyp": hyp[j]})
            i += 1; j += 1
        elif i < n and dp[i][j] == dp[i+1][j] + 1:
            ops.append({"type": "deletion", "ref": ref[i], "hyp": None})
            i += 1
        else:
            ops.append({"type": "insertion", "ref": None, "hyp": hyp[j]})
            j += 1
    return ops


## 5. Transcripció i alineació de tots els àudios

Carreguem el model Whisper i el text de referència i, per a cada àudio, el transcrivim, n'extraiem les marques temporals de cada paraula, l'alineem amb el text de referència i desem el resultat en un CSV per alumne.

In [ ]:
import pandas as pd

# --- Configuració ---
AUDIOS_FOLDER = "./audios"
RESULTS_FOLDER = "./results"
TEXT_PATH = 'text.txt'
MODEL_NAME = "large"
LANGUAGE = "es"
Z_TPC_TH = 2.0

if not os.path.exists(RESULTS_FOLDER):
    os.makedirs(RESULTS_FOLDER)

# --- Processament ---

print("Carregant model...")
model = whisper.load_model(MODEL_NAME)

with open(TEXT_PATH, "r", encoding="utf-8") as f:
    paraules_obj = normalitza_text(f.read())

for filename in os.listdir(AUDIOS_FOLDER):
    if not filename.endswith((".mp3", ".wav", ".m4a")): continue

    audio_path = os.path.join(AUDIOS_FOLDER, filename)
    print(f"\nAnalitzant: {filename}")
    print("Carregant l'àudio...")
    audio = whisper.load_audio(audio_path)
    print("Transcribint...")
    result = whisper.transcribe( model,audio,language=LANGUAGE,vad=True)
    # 2. Extreure dades temporals de Whisper
    words_whisper = []
    for seg in result["segments"]:
        for w in seg.get("words", []):
            words_whisper.append({
                "word": w["text"].strip(),
                "start": w["start"],
                "end": w["end"],
                "conf": w.get("confidence")
            })

    # 3. Alineació
    paraules_llegides = normalitza_text(result["text"])
    ops = align_words_tokens(paraules_obj, paraules_llegides)

    # 4. Fusionar Alineació amb Timestamps
    final_data = []
    w_idx = 0
    for op in ops:
        entry = {
            "type": op["type"],
            "ref": op["ref"],
            "hyp": op["hyp"],
            "start": None, "end": None, "total": 0, "conf": None
        }
        if op["hyp"] is not None and w_idx < len(words_whisper):
            w = words_whisper[w_idx]
            entry.update({
                "start": w["start"], "end": w["end"],
                "total": w["end"] - w["start"], "conf": w["conf"]
            })
            w_idx += 1
        final_data.append(entry)

    # 7. Guardar CSV individual de paraules
    final_data = pd.DataFrame(final_data)
    output_path = os.path.join(RESULTS_FOLDER, f"{os.path.splitext(filename)[0]}.csv")
    final_data.to_csv(output_path, index=False, encoding="utf-8")
    print(f"✓ Creat: {output_path}")